# Gifted-masking — review & research notebook

One place to **review** every test and model rung (M0–M4), the article experiments (E01–E06), and the
robustness suite, and to **research** with small, reproducible simulations on the maintained package.

The notebook is read-only with respect to the repository: it imports `src/gifted_masking`, reads
`results/` and `configs/`, and never writes artifacts or changes model semantics.

**How to open** (pick the `venv314` kernel, which is the workspace venv AGENTS.md requires)

```bash
/Users/omer/workspace/venvs/venv314/bin/python -m jupyter lab notebooks/review_and_research.ipynb
```

**Layout** (AGENTS.md asks to keep these concerns separate, so each section is tagged with one):

| § | Section | Concern |
|---|---------|---------|
| 1 | Setup & provenance | reproducibility |
| 2 | Inventory: models, experiments, configs, artifacts | code |
| 3 | Test review: catalogue, requirement map, runner | code correctness |
| 4 | Model review M0–M4: live spec checks, small simulations, article results | mathematical correctness → statistical robustness |
| 5 | Robustness suite | statistical robustness |
| 6 | Provenance audit: manifests and output hashes | reproducibility |
| 7 | Research sandbox | exploration (not evidence) |
| 8 | Reading the results | interpretation |

**Notebook-only dependencies** (already in the workspace venv; the package itself still needs only numpy):
`pandas>=2` for tables, `matplotlib>=3.8` for plots, `jupyterlab` to run it. Alternative avoided: stdlib
`csv` tables, which make interactive filtering much clumsier.

**Cost.** Defaults run in a few minutes. Simulations use small `n`/`T` (AGENTS.md rule 7: no large
sweeps). The full raw CSVs (up to ~390 MB) are only read on request, in §7.

In [ ]:
# --- Controls -------------------------------------------------------------
RUN_FAST_TESTS = True     # model-level unit tests (~5 s)
RUN_ALL_TESTS = False     # every test file, incl. M2 optimal-policy, article, publication (~3 min)
VERIFY_HASHES = True      # recompute sha256 of manifest outputs
MAX_HASH_MIB = 50         # skip hashing files larger than this (size is still compared)

SIM_N = 200               # children per simulated cell (small on purpose)
SIM_T = 1000              # horizon for sandbox simulations
SEEDS = (0, 1, 2)         # independent master seeds: uncertainty is between seeds

In [ ]:
import sys, json, platform, time, inspect
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def _find_root():
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if (cand / "pyproject.toml").is_file() and (cand / "src" / "gifted_masking").is_dir():
            return cand
    raise FileNotFoundError("run this notebook from inside the gifted-masking repository")

ROOT = _find_root()
EXPECTED_PYTHON = "/Users/omer/workspace/venvs/venv314/bin/python"
if Path(sys.executable).resolve() != Path(EXPECTED_PYTHON).resolve():
    print(f"WARNING: kernel is {sys.executable}; AGENTS.md requires {EXPECTED_PYTHON} (kernel 'venv314')")
for extra in (ROOT / "src", ROOT / "notebooks"):
    if str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

import review_lib as R
from gifted_masking import core
from gifted_masking.models import m0, m1, m2, m3, m4
from gifted_masking.analysis import m2_optimal as m2o
from gifted_masking.params import DEFAULT_PAYOFFS, AgentParams

RESULTS = ROOT / "results"
SUMMARIES = RESULTS / "summaries"
TESTS = ROOT / "tests"
CONFIGS = ROOT / "configs" / "frontiers_article"

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

def table(rows, **kw):
    return pd.DataFrame(rows, **kw)

def show_checks(rows, title):
    df = table(rows)
    n_fail = int((~df["passed"]).sum())
    display(Markdown(f"**{title}** — {len(df) - n_fail}/{len(df)} passed" + ("" if n_fail == 0 else f" — **{n_fail} FAILED**")))
    display(df.style.apply(lambda col: ["" if v else "background-color:#f8d7da" for v in col], subset=["passed"]))
    return df

def summary(name):
    path = SUMMARIES / name
    if not path.is_file():
        display(Markdown(f"_missing artifact `{path.relative_to(ROOT)}` — run the experiment or `./scripts/verify.sh` first_"))
        return None
    return pd.read_csv(path)

def seed_mean(df, by, value):
    # mean and between-master-seed standard error
    g = df.groupby(by)[value]
    out = g.agg(mean="mean", sd="std", n_seeds="count").reset_index()
    out["se"] = out["sd"] / np.sqrt(out["n_seeds"])
    return out

ALL_CHECKS = {}

## 1 · Setup & provenance
Record exactly which code and environment produced what you see below.

In [ ]:
info = {
    **R.git_info(ROOT),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "repo_root": ROOT.name,
    "default_payoffs": DEFAULT_PAYOFFS,
    "default_agent": AgentParams().__dict__,
}
table(info.items(), columns=["field", "value"])

## 2 · Inventory

### 2.1 Model API
Every public function/class in the model ladder, the M2 optimal-policy analysis and the shared core.

In [ ]:
api = table(R.model_api())
display(api.groupby("module").size().rename("public objects").to_frame().T)
api

### 2.2 Simulation defaults side-by-side
A quick consistency review: payoffs and learner defaults should agree across rungs unless a difference is
intentional (e.g. `n`/`horizon` differ by model).

In [ ]:
sim_fns = {
    "M0.simulate": m0.simulate, "M1.simulate": m1.simulate, "M2.simulate": m2.simulate,
    "M3.simulate (flat 2K)": m3.simulate, "M3.hierarchical": m3.simulate_hierarchical,
    "M3.global_mask": m3.simulate_global_mask, "M3.fixed_good_pond": m3.simulate_fixed_good_pond,
    "M4.simulate": m4.simulate, "M4.hysteresis": m4.hysteresis_experiment,
}
defaults = pd.DataFrame(R.simulate_defaults(sim_fns))
shared = ["b", "a", "c", "r", "alpha", "temp", "lam", "q0", "n", "horizon", "final_window_fraction", "seed"]
order = shared + [k for k in defaults.index if k not in shared]
defaults = defaults.reindex(order).fillna("—")

def _flag_row(row):
    vals = {str(v) for v in row if v != "—"}
    return ["background-color:#fff3cd" if len(vals) > 1 and row.name in shared[:8] else "" for _ in row]

display(Markdown("Rows highlighted in yellow: a shared payoff/learner default that differs between rungs."))
defaults.style.apply(_flag_row, axis=1)

### 2.3 Article experiments and configs
Auto-discovered from `results/manifests/*.json` and `configs/frontiers_article/*.json`.

In [ ]:
MANIFESTS = R.load_manifests(RESULTS)
manifests = table(MANIFESTS).drop(columns="_manifest")
display(manifests)

cfg_rows = []
for path in sorted(CONFIGS.glob("*.json")):
    cfg = json.loads(path.read_text())
    cfg_rows.append({
        "config": path.name, "experiment_id": cfg.get("experiment_id"), "model": cfg.get("model"),
        "tier": cfg.get("tier"), "run_id": cfg.get("run_id"),
        "approved_standard": cfg.get("approved_standard"),
        "master_seeds": len(cfg.get("master_seeds", [cfg.get("master_seed")] if cfg.get("master_seed") else [])),
        "has_manifest": any(m["run_id"] == cfg.get("run_id") for m in MANIFESTS),
    })
configs = table(cfg_rows)
configs

### 2.4 Result artifacts on disk
`results/` is git-ignored, so this shows what has been generated locally.

In [ ]:
artifacts = table(R.list_artifacts(RESULTS)) if RESULTS.exists() else table([])
if len(artifacts):
    display(artifacts.groupby("kind").agg(files=("path", "count"), mib=("mib", "sum")).round(1).T)
    display(artifacts.sort_values("mib", ascending=False).head(10))

## 3 · Test review

### 3.1 Catalogue
Every `test_*` method, parsed statically (no imports), with the project modules each file imports.
Filter it like any DataFrame, e.g. `catalog[catalog.file == "test_m2.py"]`.

In [ ]:
CATALOG = R.catalog_tests(TESTS)
catalog = table(CATALOG).drop(columns="_body")
per_file = catalog.groupby("file").agg(tests=("test", "count"), assertions=("assertions", "sum"),
                                       modules=("project_modules", "first"))
display(Markdown(f"**{len(catalog)} tests in {catalog.file.nunique()} files**"))
per_file

In [ ]:
# Which test files exercise which project modules
mod_matrix = (catalog.drop_duplicates("file")
              .assign(module=lambda d: d.project_modules.str.split(", "))
              .explode("module").query("module != ''")
              .pivot_table(index="file", columns="module", values="test", aggfunc="count", fill_value=0)
              .astype(bool).replace({True: "●", False: ""}))
mod_matrix

### 3.2 AGENTS.md requirements → candidate tests
A keyword index from the validation targets in AGENTS.md to tests that probably address them.
**It's a heuristic:** a hit means "look here", and zero hits means "probably untested, so check by hand".

In [ ]:
cov = table(R.requirement_coverage(CATALOG))
cov.style.apply(lambda col: ["background-color:#f8d7da" if v == 0 else "" for v in col], subset=["candidate_tests"])

### 3.3 Run tests
Fast model-level tests (`R.FAST_TEST_FILES`, ~5 s) run by default. Set `RUN_ALL_TESTS = True` for the
full suite (~3 min; `test_m2_optimal_policy.py` alone takes ~80 s). To rerun a subset: `R.run_tests(TESTS, ["test_m2.py"], name_filter="transition")`.
The canonical verifier is still `./scripts/verify.sh`; this runner is for review.

In [ ]:
selected = None if RUN_ALL_TESTS else (list(R.FAST_TEST_FILES) if RUN_FAST_TESTS else [])
if selected == []:
    display(Markdown("_tests skipped (RUN_FAST_TESTS = RUN_ALL_TESTS = False)_"))
    test_results = table([], columns=["test_id", "status", "seconds", "message"])
else:
    t0 = time.perf_counter()
    test_results = table(R.run_tests(TESTS, selected))
    display(Markdown(f"Ran **{len(test_results)}** tests in {time.perf_counter() - t0:.1f}s"))
    display(test_results.status.value_counts().to_frame().T)
    test_results["file"] = test_results.test_id.str.split(".").str[1]
    display(test_results.pivot_table(index="file", columns="status", values="test_id", aggfunc="count", fill_value=0))
    failures = test_results[~test_results.status.isin(["pass", "skip", "xfail"])]
    display(failures if len(failures) else Markdown("✅ no failures or errors"))

In [ ]:
# Slowest tests: candidates for a separate slow tier
test_results.sort_values("seconds", ascending=False).head(10) if len(test_results) else None

## 4 · Model review

Each rung has three parts:
1. **Live spec checks**: the AGENTS.md validation targets, recomputed from the package functions.
2. **Small simulation**: `SIM_N` children × `SIM_T` steps × `SEEDS`. Error bars are ±1 SE **between master seeds**.
3. **Article results**: the frozen E0x summaries for that rung, if they exist under `results/summaries`.

Payoffs throughout use the defaults `b=1, a=0.6, c=0.25, r=2` unless stated.

### 4.0 Shared core: softmax and loss aversion

In [ ]:
q_big = np.array([[1e6, 0.0], [0.0, 0.0], [-1e6, 1e6]])
probs = core.softmax_probabilities(q_big, 0.25)
R_ = np.array([-1.0, 0.0, 0.5])
rng_a, rng_b = core.make_rng(123), core.make_rng(123)
ALL_CHECKS["core"] = show_checks([
    R.check("softmax rows sum to 1 under huge Q", probs.sum(axis=1), np.ones(3)),
    R.check("softmax finite under huge Q", bool(np.isfinite(probs).all()), True),
    R.check("softmax equal Q → uniform", probs[1], [0.5, 0.5]),
    R.check("loss aversion λ=2 doubles negative reward", core.subjective_value(R_, 2.0)[0], -2.0),
    R.check("loss aversion leaves R=0 unchanged", core.subjective_value(R_, 2.0)[1], 0.0),
    R.check("loss aversion leaves R>0 unchanged", core.subjective_value(R_, 2.0)[2], 0.5),
    R.check("λ=1 is identity", core.subjective_value(R_, 1.0), R_),
    R.check("same seed → same softmax draws",
            core.softmax_sample(np.zeros((50, 3)), 0.25, rng_a), core.softmax_sample(np.zeros((50, 3)), 0.25, rng_b)),
    R.check("final window: T=1000, 0.2 → 200 steps", core.final_window_size(1000, 0.2), 200),
    R.check("final window never empty: T=3, 0.1 → 1", core.final_window_size(3, 0.1), 1),
], "Core")

### 4.1 M0: two-armed masking/authenticity bandit
$E[\text{mask}] = b-c$, $E[\text{auth}] = a + pb - (1-p)r$, objective indifference $r^*(p) = \frac{a+c}{1-p} - b$.

In [ ]:
b, a, c, r = 1.0, 0.6, 0.25, 2.0
p_star = 2.15 / 3
ALL_CHECKS["M0"] = show_checks([
    R.check("E[mask] = b - c = 0.75", m0.expected_rewards(0.5)[0], 0.75),
    R.check("p=0 → E[auth] = a - r", m0.expected_rewards(0.0)[1], a - r),
    R.check("p=1 → E[auth] = b + a", m0.expected_rewards(1.0)[1], b + a),
    R.check("E[auth](0.4) = -1.4 + 3p", m0.expected_rewards(0.4)[1], -1.4 + 3 * 0.4),
    R.check("objective p* ≈ 0.7166667 has zero gap", m0.expected_gap(p_star, b, a, c, r), 0.0, tol=1e-12),
    R.check("r*(p*) = 2", m0.objective_indifference_r(p_star), 2.0, tol=1e-12),
    R.check("λ=1 subjective r* = objective r*", m0.subjective_indifference_r(0.3, 1.0), m0.objective_indifference_r(0.3), tol=1e-12),
], "M0 analytic")

In [ ]:
p_grid = np.linspace(0.05, 0.95, 10)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
pp = np.linspace(0, 1, 101)
ax.plot(pp, np.full_like(pp, b - c), label="E[mask]")
ax.plot(pp, [m0.expected_rewards(x)[1] for x in pp], label="E[auth]")
ax.axvline(p_star, ls="--", c="grey"); ax.text(p_star + 0.01, -1.2, f"p*={p_star:.3f}")
ax.set(xlabel="acceptance p", ylabel="expected reward", title="M0 expected rewards"); ax.legend()

ax = axes[1]
pp = np.linspace(0, 0.95, 200)
ax.plot(pp, m0.objective_indifference_r(pp), label="objective r*(p)", lw=2)
for lam in (0.5, 2.5):
    ax.plot(pp, m0.subjective_indifference_r(pp, lam), ls="--", label=f"subjective, λ={lam}")
ax.set(xlabel="p", ylabel="rejection penalty r", ylim=(0, 6), title="Indifference curves (mask above)"); ax.legend()

ax = axes[2]
m0_rows = []
for lam in (0.5, 1.0, 2.5):
    rows = R.sweep(m0.simulate, "p", p_grid, SEEDS, n=SIM_N, horizon=SIM_T, lam=lam)
    df = table(rows); df["lam"] = lam; m0_rows.append(df)
    ax.errorbar(df.p, df["mean"], yerr=df.se_between_seeds, marker="o", capsize=3, label=f"λ={lam}")
ax.axvline(p_star, ls="--", c="grey")
ax.set(xlabel="p", ylabel="final-window mask rate", title=f"M0 learned masking (n={SIM_N}, T={SIM_T})"); ax.legend()
plt.tight_layout(); plt.show()
M0_SWEEP = pd.concat(m0_rows)

**Review prompt:** learned masking stays high to the right of `p*`, where authenticity is objectively
better. That is the "hot stove" effect E01 tests. Compare the simulated curves with the E01 frozen results below.

In [ ]:
e01 = summary("frontiers_hot_stove_v1_cells.csv")
if e01 is not None:
    d = e01[e01.stratum == "all"]
    agg = seed_mean(d, ["condition", "p", "lambda"], "mean_final_window_mask_rate")
    agg["cell"] = agg.condition.str.replace("_", "\n") + "\np=" + agg.p.astype(str)
    cells_order = list(dict.fromkeys(agg.sort_values(["condition", "p"]).cell))
    x = {cell: i for i, cell in enumerate(cells_order)}
    lams = sorted(agg["lambda"].unique())
    fig, ax = plt.subplots(figsize=(1.3 * len(cells_order) + 2, 4))
    for j, lam in enumerate(lams):
        g = agg[agg["lambda"] == lam]
        ax.errorbar([x[cell] + (j - 1) * 0.15 for cell in g.cell], g["mean"], yerr=g.se, fmt="o", capsize=3, label=f"λ={lam}")
    ax.set_xticks(range(len(cells_order)), cells_order, fontsize=8)
    ax.set(ylabel="final-window mask rate", ylim=(0, 1.05),
           title=f"E01 hot-stove (standard): mean ± SE across master seeds (authenticity objectively better when p > p*={p_star:.3f})")
    ax.legend(); plt.tight_layout(); plt.show()
    display(summary("frontiers_hot_stove_v1_effects.csv").query("level != 'master_seed'").head(20))

### 4.2 M1: graded disclosure
$E(d) = b - c + d\,[a + c - (1-p)(b+r)]$ is linear in $d$, so an interior disclosure level is
never strictly optimal except at indifference. Any "ladder climbing" must be an exploration effect, not static optimality.

In [ ]:
grid = m1.disclosure_levels(5)
checks = [
    R.check("grid includes 0 and 1", [grid[0], grid[-1]], [0.0, 1.0]),
    R.check("E(0) = b - c", float(m1.expected_reward(0.0, 0.6)), b - c),
    R.check("E(1) = M0 E[auth]", float(m1.expected_reward(1.0, 0.6)), m0.expected_rewards(0.6)[1], tol=1e-12),
    R.check("acceptance(d=1) = p", m1.acceptance_probability(0.3, np.array([1.0]))[0], 0.3),
    R.check("acceptance(d=0) = 1", m1.acceptance_probability(0.3, np.array([0.0]))[0], 1.0),
]
interior_strict = []
for p in np.linspace(0, 1, 41):
    e = m1.expected_reward(grid, p)
    interior_strict.append(e[1:-1].max() > max(e[0], e[-1]) + 1e-12)
checks.append(R.check("no interior d strictly optimal on a 41-point p grid", any(interior_strict), False))
ALL_CHECKS["M1"] = show_checks(checks, "M1 analytic")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
dd = np.linspace(0, 1, 50)
for p in (0.3, 0.6, p_star, 0.9):
    axes[0].plot(dd, m1.expected_reward(dd, p), label=f"p={p:.2f}")
axes[0].set(xlabel="disclosure d", ylabel="E(d)", title="M1 expected reward is linear in d"); axes[0].legend()

for levels in (2, 5):
    df = table(R.sweep(m1.simulate, "p", p_grid, SEEDS, n=SIM_N, horizon=SIM_T, levels=levels))
    axes[1].errorbar(df.p, df["mean"], yerr=df.se_between_seeds, marker="o", capsize=3, label=f"{levels} levels")
axes[1].axvline(p_star, ls="--", c="grey")
axes[1].set(xlabel="p", ylabel="final-window mean disclosure", title="M1 learned disclosure"); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
e02 = summary("frontiers_m1_binary_graded_v1_cells.csv")
if e02 is not None:
    d = e02[e02.stratum == "all"]
    metrics = ["mean_final_window_mean_disclosure", "mean_final_window_full_authentic_rate", "probability_final_window_optimal_endpoint_modal"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
    for ax, metric in zip(axes, metrics):
        for arch, g in seed_mean(d, ["architecture", "p"], metric).groupby("architecture"):
            ax.errorbar(g.p, g["mean"], yerr=g.se, marker="o", capsize=3, label=arch)
        ax.set(title=metric.replace("_", " "), xlabel="p")
    axes[0].legend(); fig.suptitle("E02 binary vs graded disclosure (standard)"); plt.tight_layout(); plt.show()
    display(summary("frontiers_m1_binary_graded_v1_arms.csv"))

### 4.3 M2: stigma-state MDP
States: neutral / accepted / stigmatized. Transition order (from `m2_optimal.TRANSITION_ORDERING`):
reward first, then δ-turnover or η-escape act on the **next** state. Masking is called "suboptimal" only
against the value-iteration optimum.

In [ ]:
def one_step(state, action, outcome=0.0, turnover=0.99, escape=0.99, p=0.5, eta=0.1):
    R_, s_next = m2.transition_step(np.array([state]), np.array([action]), p=p, eta=eta,
                                    outcome_uniform=np.array([outcome]), turnover_uniform=np.array([turnover]),
                                    escape_uniform=np.array([escape]))
    return float(R_[0]), int(s_next[0])

N_, A_, S_ = m2.STATE_NEUTRAL, m2.STATE_ACCEPTED, m2.STATE_STIGMATIZED
checks = [
    R.check("neutral + mask → stays neutral, b-c", one_step(N_, 0), (b - c, N_)),
    R.check("neutral + auth accepted → accepted, b+a", one_step(N_, 1, outcome=0.1), (b + a, A_)),
    R.check("neutral + auth rejected → stigmatized, a-r", one_step(N_, 1, outcome=0.9), (a - r, S_)),
    R.check("accepted + auth → reliable b+a", one_step(A_, 1, outcome=0.99), (b + a, A_)),
    R.check("accepted + δ turnover → neutral next", one_step(A_, 1, turnover=0.0)[1], N_),
    R.check("stigmatized + mask → b2-c = 0.25", one_step(S_, 0), (0.5 - c, S_)),
    R.check("stigmatized redemption uses p/2 (u=0.2<0.25)", one_step(S_, 1, outcome=0.2), (b + a, A_)),
    R.check("stigmatized fail uses r2 (u=0.3≥0.25)", one_step(S_, 1, outcome=0.3), (a - 2.5, S_)),
    R.check("stigmatized + η escape → neutral next", one_step(S_, 0, escape=0.0)[1], N_),
]
sol = m2o.solve_m2_value_iteration(m2o.M2Params(p=0.5, eta=0.1))
checks += [
    R.check("value iteration converged", bool(sol["converged"]), True),
    R.check("Bellman residual < 1e-9", float(sol["bellman_residual"]) < 1e-9 if "bellman_residual" in sol else True, True),
]
ALL_CHECKS["M2"] = show_checks(checks, "M2 transitions & value iteration")
display(Markdown("Optimal Q at p=0.5, η=0.1 (objective rewards):"))
pd.DataFrame(sol["Q"], index=m2o.STATE_NAMES, columns=m2o.ACTION_NAMES)

In [ ]:
# Objective optimal neutral action vs the static M0 comparator across the (p, η) grid
ps = np.round(np.arange(0, 1.0001, 0.05), 3)
etas = np.round(np.arange(0, 0.8001, 0.05), 3)
region_code = {"static & dynamic: mask": 0, "dynamic authenticity overrides static mask": 1,
               "static & dynamic: authentic": 2, "dynamic mask overrides static authentic": 3}
grid_rows = []
for eta in etas:
    for p in ps:
        prm = m2o.M2Params(p=float(p), eta=float(eta))
        dyn = m2o.solve_m2_value_iteration(prm)["chosen_actions"]["neutral"]
        stat = m2o.static_m0_comparator(prm)["optimal_action"]
        label = ("static & dynamic: mask" if dyn == stat == "mask" else
                 "static & dynamic: authentic" if dyn == stat == "authentic" else
                 "dynamic authenticity overrides static mask" if dyn == "authentic" else
                 "dynamic mask overrides static authentic")
        grid_rows.append({"p": p, "eta": eta, "dynamic": dyn, "static": stat, "region": label})
m2_grid = table(grid_rows)
thr = [(eta, m2o.dynamic_authenticity_threshold(m2o.M2Params(p=0.5, eta=float(eta)), tol=1e-6)) for eta in etas]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
mat = m2_grid.pivot(index="eta", columns="p", values="region").replace(region_code).astype(float)
cmap = matplotlib.colors.ListedColormap(["#9e9e9e", "#4c9be8", "#2e7d32", "#c62828"])
axes[0].imshow(mat.values, origin="lower", aspect="auto", cmap=cmap, vmin=-0.5, vmax=3.5,
               extent=[ps[0] - 0.025, ps[-1] + 0.025, etas[0] - 0.025, etas[-1] + 0.025])
axes[0].plot([t for _, t in thr], [e for e, _ in thr], "w--", lw=1.5)
axes[0].axvline(p_star, c="k", lw=0.8, ls=":")
handles = [matplotlib.patches.Patch(color=cmap(i), label=k) for k, i in region_code.items()]
axes[0].legend(handles=handles, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2)
axes[0].set(xlabel="p", ylabel="η (stigma escape)", title="M2 objective optimum (neutral) vs static M0\nwhite dashes: VI threshold; dotted: static p*")

axes[1].plot([e for e, _ in thr], [t for _, t in thr], marker="o")
axes[1].axhline(p_star, ls=":", c="k", label="static M0 p*")
axes[1].set(xlabel="η", ylabel="dynamic authenticity threshold p", title="Value-iteration threshold vs η"); axes[1].legend()
plt.tight_layout(); plt.show()
m2_grid.region.value_counts().to_frame()

In [ ]:
# Learned vs optimal at one cell: classify with the package's own classifier
P_CELL, ETA_CELL = 0.55, 0.0
diag = m2.simulate_with_diagnostics(P_CELL, ETA_CELL, n=SIM_N, horizon=SIM_T, seed=SEEDS[0])
cls = m2o.classify_m2_cell(m2o.M2Params(p=P_CELL, eta=ETA_CELL), diag)
display(table([{
    "p": P_CELL, "eta": ETA_CELL, "region": cls["region"],
    "static": cls["static_neutral_action"], "dynamic optimum": cls["dynamic_neutral_action"],
    "learned (neutral)": cls["learned_neutral_action"], "regret_neutral": cls["regret_neutral"],
    "mean mask rate": float(diag["mask_rate_final"].mean()),
}]))
display(Markdown("Learned mask probability by state (final window):"))
table([cls["learned"]["learned_mask_probabilities"]])

In [ ]:
e03_stable = summary("frontiers_m2_regime_replication_v1_stable_cells.csv")
if e03_stable is not None:
    lab = e03_stable.assign(cls=np.where(e03_stable.stable, e03_stable.classification, "unstable"))
    codes = {k: i for i, k in enumerate(sorted(lab.cls.unique()))}
    mat = lab.pivot(index="eta", columns="p", values="cls").replace(codes).astype(float)
    cm = matplotlib.colors.ListedColormap(["#c62828", "#2e7d32", "#bdbdbd", "#ffb300"][: len(codes)])
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.imshow(mat.values, origin="lower", aspect="auto", cmap=cm, vmin=-0.5, vmax=len(codes) - 0.5,
              extent=[mat.columns.min() - 0.025, mat.columns.max() + 0.025, mat.index.min() - 0.025, mat.index.max() + 0.025])
    ax.plot([t for _, t in thr], [e for e, _ in thr], "w--", lw=1.5, label="objective VI threshold (§4.3)")
    ax.legend(handles=[matplotlib.patches.Patch(color=cm(i), label=k) for k, i in codes.items()]
              + [matplotlib.lines.Line2D([], [], c="w", ls="--", label="objective VI threshold (§4.3)")],
              fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1), facecolor="#eeeeee")
    ax.set(xlabel="p", ylabel="η", title="E03 standard: learner vs objective optimum (stable = ≥4/5 seeds)")
    plt.tight_layout(); plt.show()
    display(summary("frontiers_m2_regime_replication_v1_thresholds.csv"))
    display(summary("frontiers_m2_regime_replication_v1_effects.csv"))

### 4.4 M3: peer-group / context search (+ BFLPE)
One good pond (`p_good=0.85`) among K groups. The flat formulation has 2K arms: arm → (group = arm // 2,
authentic = arm % 2). BFLPE comparison cost `bf` applies only to accepted authenticity in the good pond.
Analytic crossover: $bf^* = [p_g(b+a) + (1-p_g)(a-r) - (b-c)] / p_g$.

In [ ]:
K = 3
arms = np.arange(2 * K)
grp, auth = m3.decode_action(arms)
display(table({"arm": arms, "group": grp, "authentic": auth,
               "E[reward]": m3.expected_reward_for_action(arms, k_groups=K).round(4)}).set_index("arm").T)
p_good = 0.85
bf_star = (p_good * (b + a) + (1 - p_good) * (a - r) - (b - c)) / p_good
ALL_CHECKS["M3"] = show_checks([
    R.check("decode: arm 5 → group 2, authentic", (int(m3.decode_action(5)[0]), bool(m3.decode_action(5)[1])), (2, True)),
    R.check("group_probabilities K=4", m3.group_probabilities(4, 0.85, 0.2), [0.85, 0.2, 0.2, 0.2]),
    R.check("masking payoff identical across groups", np.ptp(m3.expected_reward_for_action(np.arange(0, 2 * K, 2), k_groups=K)), 0.0),
    R.check("bf* ≈ 0.470588", bf_star, 0.470588, tol=1e-6),
    R.check("at bf*, good-pond auth = mask", float(m3.expected_reward_for_action(1, k_groups=K, bf=bf_star)), b - c, tol=1e-12),
    R.check("bf does not affect bad-pond auth", float(m3.expected_reward_for_action(3, k_groups=K, bf=0.3)),
            float(m3.expected_reward_for_action(3, k_groups=K, bf=0.0))),
], "M3 analytic")

In [ ]:
# Action-space ablation (AGENTS.md: "larger K causes masking" must survive these)
architectures = {
    "flat_2k": lambda k, **kw: m3.simulate(k, **kw),
    "hierarchical": lambda k, **kw: m3.simulate_hierarchical(k, **kw),
    "global_mask": lambda k, **kw: m3.simulate_global_mask(k, **kw),
}
Ks = (2, 4, 8)
abl = []
for name, fn in architectures.items():
    for k in Ks:
        for metric, idx in (("settled_good_auth", 0), ("mask_rate", 1)):
            s = R.replicate(lambda seed, **kw: fn(k, seed=seed, **kw)[idx], SEEDS, n=SIM_N, horizon=SIM_T)
            abl.append({"architecture": name, "K": k, "metric": metric, **s})
fixed = {m: R.replicate(lambda seed, **kw: m3.simulate_fixed_good_pond(seed=seed, **kw)[i], SEEDS, n=SIM_N, horizon=SIM_T)
         for m, i in (("settled_good_auth", 0), ("mask_rate", 1))}
abl = table(abl)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric in zip(axes, ("settled_good_auth", "mask_rate")):
    for name, g in abl[abl.metric == metric].groupby("architecture"):
        ax.errorbar(g.K, g["mean"], yerr=g.se_between_seeds, marker="o", capsize=3, label=name)
    ax.axhline(fixed[metric]["mean"], ls="--", c="k", label="fixed known good pond")
    ax.set_xscale("log", base=2); ax.set_xticks(Ks, labels=[str(k) for k in Ks]); ax.minorticks_off()
    ax.set(xlabel="K groups", title=f"{metric} (n={SIM_N}, T={SIM_T})")
axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
e04 = summary("frontiers_m3_search_architectures_v1_curves.csv")
if e04 is not None:
    metrics = ["mean_settled_any_good_auth_argmax", "mean_final_window_mask_rate", "mean_expected_static_regret"]
    modes = sorted(e04.canonical_identity_mode.unique())
    fig, axes = plt.subplots(len(modes), 3, figsize=(15, 3.6 * len(modes)), squeeze=False)
    for i, mode in enumerate(modes):
        for j, metric in enumerate(metrics):
            ax = axes[i, j]
            sub = e04[(e04.canonical_identity_mode == mode) | (e04.architecture == "fixed_known_good")]
            for arch, g in seed_mean(sub, ["architecture", "K"], metric).groupby("architecture"):
                if arch == "fixed_known_good":
                    ax.axhline(g["mean"].iloc[0], ls="--", c="k", label=arch)
                else:
                    ax.errorbar(g.K, g["mean"], yerr=g.se, marker="o", capsize=3, label=arch)
            ks = sorted(sub.K.unique())
            ax.set_xscale("log", base=2); ax.set_xticks(ks, labels=[str(k) for k in ks]); ax.minorticks_off()
            ax.set(title=f"{metric.replace('mean_', '')}\nidentity={mode}", xlabel="K")
    axes[0, 0].legend(fontsize=8); fig.suptitle("E04 search architectures (standard)"); plt.tight_layout(); plt.show()

In [ ]:
e05 = summary("frontiers_m3_bflpe_crossover_v1_cells.csv")
if e05 is not None:
    fig, ax = plt.subplots(figsize=(7, 4))
    for arch, g in seed_mean(e05, ["architecture", "bf"], "modal_good_auth_argmax_fraction").groupby("architecture"):
        ax.errorbar(g.bf, g["mean"], yerr=g.se, marker=".", capsize=2, label=arch)
    ax.axvline(bf_star, ls="--", c="k", label=f"analytic bf*={bf_star:.3f}"); ax.axhline(0.5, c="grey", lw=0.8)
    ax.set(xlabel="bf (comparison cost)", ylabel="fraction modal good-pond authentic",
           title="E05 BFLPE crossover (standard)"); ax.legend(); plt.tight_layout(); plt.show()
    tr = summary("frontiers_m3_bflpe_crossover_v1_transitions.csv")
    display(tr[["architecture", "level", "master_seed", "transition_bf", "transition_status", "ci_low", "ci_high",
                "analytic_threshold_bf", "learned_minus_analytic_threshold", "decision_eligible"]])

### 4.5 M4: endogenous receptivity
$p_t$ rises by `ka·(p_max−p)` after accepted authenticity, falls by `kr·(p−p_min)` after rejection and by
`km·(p−p_min)` after masking. Reward first, then the $p_t$ update. Q-values don't condition on $p_t$, so
this is a **non-stationary/endogenous bandit**, not stateful Q-learning.

In [ ]:
pmin, pmax = 0.05, 0.95
pt = np.array([0.5, 0.5, 0.5, pmin, pmax])
authentic = np.array([True, True, False, False, True])
accepted = np.array([True, False, False, False, True])
up = m4.update_receptivity(pt, authentic, accepted, km=0.1, ka=0.12, kr=0.04, p_min=pmin, p_max=pmax)
up_km0 = m4.update_receptivity(pt, authentic, accepted, km=0.0, ka=0.12, kr=0.04, p_min=pmin, p_max=pmax)
_, p_final, traj, *_ = m4.simulate(0.45, 0.002, n=SIM_N, horizon=SIM_T, ntraj=SIM_N, seed=SEEDS[0])
ALL_CHECKS["M4"] = show_checks([
    R.check("accepted auth: p += ka(p_max - p)", up[0], 0.5 + 0.12 * (pmax - 0.5), tol=1e-12),
    R.check("rejected auth: p -= kr(p - p_min)", up[1], 0.5 - 0.04 * (0.5 - pmin), tol=1e-12),
    R.check("mask: p -= km(p - p_min)", up[2], 0.5 - 0.1 * (0.5 - pmin), tol=1e-12),
    R.check("km = 0 → masking leaves p unchanged", up_km0[2], 0.5),
    R.check("p at p_min stays at p_min under masking", up[3], pmin),
    R.check("p at p_max stays at p_max when accepted", up[4], pmax),
    R.check(f"simulated trajectories stay in [p_min, p_max] ({SIM_N}×{SIM_T})",
            bool((traj >= pmin - 1e-12).all() and (traj <= pmax + 1e-12).all()), True),
], "M4 receptivity")

In [ ]:
e06_cfg = json.loads((CONFIGS / "e06_m4_paths_smoke.json").read_text())
km_values = np.asarray(e06_cfg["km_values"], dtype=float)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for p0, col in ((0.15, "C3"), (0.45, "C0"), (0.85, "C2")):
    _, _, tr, *_ = m4.simulate(p0, 0.002, n=40, horizon=SIM_T, ntraj=8, seed=SEEDS[0])
    axes[0].plot(tr, c=col, alpha=0.5, lw=0.8)
    axes[0].plot([], c=col, label=f"p0={p0}")
axes[0].set(xlabel="t", ylabel="p_t", title="M4 receptivity trajectories (km=0.002)"); axes[0].legend()

hy = [m4.hysteresis_experiment(0.45, km_values, n=SIM_N, horizon=e06_cfg["segment_horizon"] * 5, seed=s) for s in SEEDS]
for ax, key in zip(axes[1:], ("mask", "p")):
    up = np.array([h[f"{key}_up"] for h in hy]); down = np.array([h[f"{key}_down"][::-1] for h in hy])
    ax.errorbar(km_values, up.mean(0), yerr=up.std(0, ddof=1) / np.sqrt(len(SEEDS)), marker="o", capsize=3, label="ramp up")
    ax.errorbar(km_values, down.mean(0), yerr=down.std(0, ddof=1) / np.sqrt(len(SEEDS)), marker="s", capsize=3, label="ramp down")
    ax.set_xticks(km_values, labels=[f"{k:g}" for k in km_values])
    ax.set(xlabel="km", ylabel=f"mean {key}", title=f"Ramp-up vs ramp-down: {key} (E06 smoke km grid)"); ax.legend()
plt.tight_layout(); plt.show()
display(Markdown(
    "**Caution (AGENTS.md):** a gap between the up and down curves shows *path dependence* in this run. "
    "It is not hysteresis until it survives a cold-start control, km-disabled controls and seed replication. "
    "That is what E06 is built to test, and E06 has **smoke-tier results only**. "
    "Check the gap at km = 0, where masking does not move receptivity: whatever gap remains there comes from "
    "extra learning time along the path, not from km."))

In [ ]:
e06_paths = summary("frontiers_m4_path_diagnostics_v1_smoke_paths.csv")
if e06_paths is not None:
    types = list(e06_paths.path_type.unique())
    fig, axes = plt.subplots(1, len(types), figsize=(4 * len(types), 3.5), sharey=True)
    for ax, pt_ in zip(np.atleast_1d(axes), types):
        g = e06_paths[e06_paths.path_type == pt_]
        ax.plot(g.km, g.mask_up, "o-", label="up"); ax.plot(g.km, g.mask_down, "s-", label="down")
        ax.set_xticks(g.km, labels=[f"{k:g}" for k in g.km], rotation=45); ax.set(title=pt_, xlabel="km")
    np.atleast_1d(axes)[0].set_ylabel("mask rate"); np.atleast_1d(axes)[0].legend()
    fig.suptitle("E06 smoke path diagnostics (decision_eligible = False)"); plt.tight_layout(); plt.show()
    display(summary("frontiers_m4_path_diagnostics_v1_smoke_loops.csv"))

## 5 · Robustness suite
From `./scripts/verify.sh` (`gifted_masking.cli robustness --suite full`): each headline effect with
bootstrap CIs over matched units, recomputed independently from raw.

In [ ]:
rob = summary("robustness_full_summary.csv")
unc = summary("robustness_full_headline_uncertainty.csv")
rec = summary("robustness_full_recomputed_effects.csv")
if rob is not None:
    display(Markdown(f"**{int(rob.passed.sum())}/{len(rob)} checks passed**"))
    display(rob)
if unc is not None:
    u = unc.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.4 * len(u) + 1))
    ax.errorbar(u.estimate, range(len(u)), xerr=[u.estimate - u.ci95_low, u.ci95_high - u.estimate],
                fmt="o", capsize=3, color="C0")
    ax.axvline(0, c="k", lw=0.8); ax.set_yticks(range(len(u)), u.effect, fontsize=8)
    ax.set(xlabel="effect estimate (95% bootstrap CI)", title="Headline effects"); plt.tight_layout(); plt.show()
if rec is not None and rob is not None:
    display(Markdown("Independent recomputation from raw (should match the summary):"))
    display(rec)

## 6 · Provenance audit
For each manifest: does the recorded config hash match its normalized config, was the run made from a
dirty tree, and do the output files on disk still hash to the recorded values?

In [ ]:
display(manifests[["manifest", "experiment_id", "tier", "config_sha_matches", "git_commit", "git_dirty", "verification", "raw_rows"]])
if VERIFY_HASHES:
    t0 = time.perf_counter()
    hash_rows = [row for m in MANIFESTS for row in R.verify_manifest_hashes(m["_manifest"], ROOT, max_mib=MAX_HASH_MIB)]
    hashes = table(hash_rows)
    display(Markdown(f"Hashed {len(hashes)} declared outputs in {time.perf_counter() - t0:.1f}s"))
    display(hashes.status.value_counts().to_frame().T)
    bad = hashes[hashes.status.isin(["MISMATCH", "missing"]) | (hashes.get("bytes_match") == False)]
    display(bad if len(bad) else Markdown("✅ every hashed output matches its manifest"))

## 7 · Research sandbox
Tools for new questions. **Sandbox output is exploration, not evidence:** anything worth claiming
needs a config, manifest and seed hierarchy like E01–E06.

* `R.sweep(fn, param, values, seeds, **fixed)`: one-at-a-time sweep with master-seed replication.
* `R.replicate(fn, seeds, select=..., **kw)`: mean ± SE between seeds for one cell.
* `R.learner_steps(n, T, cells, seeds)`: size a run before you launch it.
* `load_raw(run_id, ...)`: chunked, filtered reads of the large raw CSVs.

In [ ]:
# Example: AGENTS.md sensitivity targets for M0 at p=0.8 (authenticity objectively better)
fig, axes = plt.subplots(1, 4, figsize=(17, 3.6), sharey=True)
for ax, (param, values) in zip(axes, {"temp": [0.05, 0.1, 0.25, 0.5, 1.0], "alpha": [0.02, 0.05, 0.1, 0.3, 0.6],
                                      "lam": [0.5, 1.0, 1.5, 2.5, 4.0], "q0": [-1.0, 0.0, 0.75, 1.6, 3.0]}.items()):
    df = table(R.sweep(m0.simulate, param, values, SEEDS, p=0.8, n=SIM_N, horizon=SIM_T))
    ax.errorbar(df[param], df["mean"], yerr=df.se_between_seeds, marker="o", capsize=3)
    ax.set(xlabel=param, title=f"M0 mask rate vs {param} (p=0.8)")
axes[0].set_ylabel("final-window mask rate"); plt.tight_layout(); plt.show()
print(f"cost of that panel: {R.learner_steps(SIM_N, SIM_T, cells=20, seeds=len(SEEDS)):,} child-steps")

In [ ]:
# Example: horizon sensitivity, since "lock-in" at T may be slow convergence. Is mask rate still moving with T?
df = table(R.sweep(m0.simulate, "horizon", [250, 500, 1000, 2000, 4000], SEEDS, p=0.8, n=SIM_N))
df

In [ ]:
# Example: M2 dynamic threshold sensitivity to γ, δ and r2 (analytic, no simulation needed)
rows = []
for gamma in (0.5, 0.75, 0.9, 0.99):
    for delta in (0.0, 0.02, 0.1, 0.3):
        for r2 in (2.5, 8.0):
            t = m2o.dynamic_authenticity_threshold(m2o.M2Params(p=0.5, eta=0.0, gamma=gamma, delta=delta, r2=r2), tol=1e-6)
            rows.append({"gamma": gamma, "delta": delta, "r2": r2, "threshold_p": t})
table(rows).pivot_table(index=["r2", "gamma"], columns="delta", values="threshold_p").round(3)

In [ ]:
RAW_DIR = RESULTS / "raw"

def load_raw(run_id, *, usecols=None, where=None, nrows=None, chunksize=200_000):
    # Read results/raw/<run_id>_units.csv in chunks.
    # where: optional callable(DataFrame) -> boolean mask, applied per chunk.
    path = RAW_DIR / f"{run_id}_units.csv"
    if not path.is_file():
        raise FileNotFoundError(path.relative_to(ROOT))
    if where is None:
        return pd.read_csv(path, usecols=usecols, nrows=nrows)
    parts, kept = [], 0
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunksize):
        part = chunk[where(chunk)]
        parts.append(part); kept += len(part)
        if nrows and kept >= nrows:
            break
    out = pd.concat(parts) if parts else pd.DataFrame(columns=usecols)
    return out.head(nrows) if nrows else out

def raw_columns(run_id):
    return list(pd.read_csv(RAW_DIR / f"{run_id}_units.csv", nrows=0).columns)

# Example: per-child distribution behind one E01 cell
raw_e01 = load_raw("frontiers_hot_stove_v1", nrows=20_000)
display(Markdown(f"`frontiers_hot_stove_v1_units.csv`: {len(raw_columns('frontiers_hot_stove_v1'))} columns; first 20k rows loaded"))
raw_e01.head()

In [ ]:
# Per-child distribution: bimodality (lock-in) is hidden by cell means
num = raw_e01.select_dtypes("number")
mask_col = next((col for col in num.columns if "mask" in col and "final" in col), None)
if mask_col:
    fig, ax = plt.subplots(figsize=(8, 3.5))
    for cond, g in raw_e01.groupby("condition"):
        ax.hist(g[mask_col].dropna(), bins=40, histtype="step", lw=1.5, label=cond)
    ax.set(xlabel=mask_col, ylabel="children", title="E01 per-child final-window masking (first 20k raw rows)")
    ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

## 8 · Reading the results
The AGENTS.md acceptance questions, with where this notebook shows the evidence. The answers are yours.

| Question | Evidence here |
|---|---|
| Does the code match the mathematical specification? | §4 live checks (summary below), §3 tests |
| Are all random processes reproducible? | §4.0 seed check, `test_determinism.py`, §6 hashes |
| Are phase maps robust to seed, T, n and grid? | §4 between-seed SE, §7 horizon/learner sweeps, E03 stable cells |
| Is M2 lock-in benchmarked against optimal policies? | §4.3 VI map, `classify_m2_cell`, E03 |
| Are M3 context-search conclusions robust to action-space ablations? | §4.4 ablation, E04 |
| Is M4 hysteresis shown, or only path dependence? | §4.5 plus E06 (smoke only, not decision-eligible) |
| Are uncertainty intervals reported? | §5 headline CIs, E0x effects tables |

In [ ]:
spec = pd.concat({k: v[["check", "passed"]] for k, v in ALL_CHECKS.items()}, names=["rung"]).reset_index(level=0)
overview = spec.groupby("rung").passed.agg(passed="sum", total="count")
if len(test_results):
    overview.loc["unit tests (§3.3)"] = [int((test_results.status == "pass").sum()), len(test_results)]
if rob is not None:
    overview.loc["robustness checks (§5)"] = [int(rob.passed.sum()), len(rob)]
if VERIFY_HASHES:
    overview.loc["output hashes (§6)"] = [int((hashes.status == "match").sum()), int(hashes.status.isin(["match", "MISMATCH"]).sum())]
overview